# P6 — Generator retrain on `attribute_spec_text` (the real two-stage gate)

Open in Cursor, connect to a GPU (**A100 recommended** — this is a full 2-epoch train over the whole corpus; a T4 works but is slow), then **Run All**. It retrains the QLoRA generator with the **locked knobs unchanged**, swapping ONLY the input field `instruction → attribute_spec_text` (ground-truth spec, derived from each row's `measured_behavior` / refuse kind — no interpreter needed), then scores the **same P1 unit-aware holdout** with the same input.

This is the un-confounded version of the oracle gate: a **spec-aware** generator scored apples-to-apples against the one-stage baseline.

**Paste back**: the final `METRIC=<float>` line **and** the `{"bridge_summary": {…}}` line (it carries the adapter HF path + train summary).

**Gate**: two-stage is validated iff the honest token accuracy **≥ 0.362** (the one-stage baseline on the same holdout) within CI. The seam-injectivity analysis put the ceiling at 100%, so there is ample headroom.

Case trap: code at `/content/SLM` (UPPERCASE); staged corpus + `SLM_ARTIFACT_ROOT=/content/slm` (lowercase). HF upload uses the **write** token (`SLM_Alpha_Write`); read-only `HF_TOKEN` 403s.

In [ ]:
## CELL 1 — provision the runtime (idempotent; run first)
import os, pathlib, subprocess, sys

REPO = '/content/SLM'
BRANCH = 'feat/two-stage'
if not pathlib.Path(REPO, '.git').is_dir():
    subprocess.run(['git', 'clone', 'https://github.com/ericrcwu001/SLM', REPO], check=True)
os.chdir(REPO)
if not os.environ.get('SLM_PROVISIONED'):
    subprocess.run(['git', 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', 'checkout', BRANCH], check=True)
    subprocess.run(['git', 'pull', 'origin', BRANCH], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[sft]'], check=True)
    os.environ['SLM_PROVISIONED'] = '1'
print('HEAD:', subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout.strip())

# HF read + write tokens (write is needed to upload the retrained adapter). Prefer an uploaded .env.
def _env_token(name):
    for _p in ('/content/SLM/.env', '/content/.env', '.env'):
        fp = pathlib.Path(_p)
        if fp.is_file():
            for _l in fp.read_text().splitlines():
                s = _l.strip()
                if s.startswith(name + '='):
                    return s.split('=', 1)[1].strip().strip('"').strip("'")
    return None
import getpass
for _name, _prompt in (('HF_TOKEN', 'HF_TOKEN (read, for staging): '),
                       ('HF_WRITE_TOKEN', 'HF_WRITE_TOKEN (SLM_Alpha_Write, for adapter upload): ')):
    if not os.environ.get(_name):
        os.environ[_name] = _env_token(_name) or getpass.getpass(_prompt).strip()
print('HF_TOKEN set:', bool(os.environ.get('HF_TOKEN')), '| HF_WRITE_TOKEN set:', bool(os.environ.get('HF_WRITE_TOKEN')))

# stage the corpus ONLY if missing (~9.85GB).
os.environ['SLM_ARTIFACT_ROOT'] = '/content/slm'
if not pathlib.Path('/content/slm/tokenizer/final/model.pt').is_file():
    print('corpus missing -> staging ~9.85GB from hf://datasets/ericrcwu/LUT_SLM ...')
    subprocess.run(['slm_stage', 'stage', '--durable-root', 'hf://datasets/ericrcwu/LUT_SLM',
                    '--local-root', '/content/slm'], check=True)
else:
    print('corpus already staged at /content/slm')

In [ ]:
## CELL 2 — imports + asserts + build/reuse base_resized
import os, pathlib, shutil, json, subprocess, sys
import torch
import bitsandbytes as bnb, peft, accelerate, qwen_vl_utils
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
os.environ['SLM_ARTIFACT_ROOT'] = '/content/slm'
assert torch.cuda.is_available(), 'no CUDA — connect an A100 (recommended) or T4'
print('GPU:', torch.cuda.get_device_name(0), '| bf16:', torch.cuda.is_bf16_supported())
assert pathlib.Path('/content/slm/tokenizer/final/model.pt').is_file(), 'frozen tokenizer missing (run CELL 1)'
free_gb = shutil.disk_usage('/content').free / 1e9
need_gb = 25 if not pathlib.Path('models/base_resized/vocab_resize_manifest.json').is_file() else 5
assert free_gb > need_gb, f'need >{need_gb}GB free on /content (have {free_gb:.1f})'

D = pathlib.Path('models/base_resized')
if (D / 'vocab_resize_manifest.json').is_file() and (D / 'preprocessor_config.json').is_file():
    print('base_resized already built -> reusing.')
else:
    subprocess.run([sys.executable, '-m', 'sft.vocab_resize', '--out', 'models/base_resized'],
                   check=True, env={**os.environ, 'SLM_ARTIFACT_ROOT': '/content/slm'})
m = json.load(open(D / 'vocab_resize_manifest.json'))
assert m.get('tokenizer_version') == 'vq_v2_srgbres_17to4_cb256_t64__w91cffdd2c82f', m.get('tokenizer_version')
assert m.get('vq_codebook_sha256'), 'null vq_codebook_sha256 — identity did not bind'
print('base_resized OK; identity bound:', m['tokenizer_version'], '| free GB:', round(free_gb, 1))

In [ ]:
## CELL 3 — retrain the generator on attribute_spec_text + score the P1 holdout 🛑 paste back METRIC= + bridge_summary
# Locked knobs unchanged; the ONLY change is input_field=attribute_spec_text (configs/candidate_two_stage.json).
# --smoke-size 0 = full corpus (2 epochs); --score-limit 0 = full unit-aware holdout; uploads the adapter to HF.
import os
os.environ['PUSH_HF_REPO'] = 'ericrcwu/LUT_SLM_sft_adapters'
!SLM_ARTIFACT_ROOT=/content/slm PUSH_HF_REPO=$PUSH_HF_REPO python -m sft.bilevel_bridge --mode colab \
    --config /content/SLM/configs/candidate_two_stage.json --resized-model models/base_resized \
    --smoke-size 0 --score-limit 0 --timeout 28800 --run-id p6_twostage
# Compare the printed METRIC= (honest two-stage token accuracy on the P1 holdout) to the one-stage
# baseline 0.362. Two-stage validated iff METRIC >= 0.362 within CI (see score_summary in the log).